# Measure the final inference pipeline

Run this **after the CPU evaluation/export succeeds**. Select **T4 ×2**, Internet disabled, then **Save Version → Save & Run All**.

Attach only:
- The saved output of `03_evaluer_exporter_cpu`, containing `rsna3-inference/` with fifteen model weights, the code and offline wheels. A private dataset containing that folder also works.
- The **RSNA Knee Abnormality Detection competition**, including `train.csv`, `train_series.csv`, and the training DICOM folders.

The bootstrap and dependencies are included in the exported package. No CPU training cache, training run, or original `CPU Probes` input is needed at this stage.

The notebook copies the package into its writable output, checks its versions and checksums, and measures complete decoding, rendering and inference on a deterministic sample of fifty training studies. Sample selection does not use labels. It preserves the trained weights and ensemble recipe. An existing complete timing profile is reused.

Update the current quota below. The measured T4 ×2 quota factor remains **1.0**. The benchmark has a two-hour ceiling; its actual duration will be measured. This is a timing check, not a competition submission.

After success, keep the new `rsna3-inference/` folder from **this GPU notebook's outputs**. Its manifest contains the timing profile required by `04_soumettre.ipynb`; the earlier CPU-only export does not.


In [ ]:
# Current remaining quota; update if Kaggle now displays a different value.
QUOTA_AVAILABLE = 14.0
MEASURED_QUOTA_FACTOR = 1.0  # T4 x2 pilot: three quota minutes for three elapsed minutes.
MAX_BENCHMARK_HOURS = 2.0


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys
BUNDLE_ROOT = None
if BUNDLE_ROOT is None:
    choices = []
    for p in Path('/kaggle/input').rglob('manifest.json'):
        try:
            if json.loads(p.read_text()).get('format') == 'rsna3-full-models-v1':
                choices.append(p.parent)
        except (OSError, ValueError):
            pass
    if len(choices) != 1:
        raise ValueError(f'Choisir BUNDLE_ROOT parmi {choices}')
    BUNDLE_ROOT = choices[0]
BUNDLE_ROOT = Path(BUNDLE_ROOT)
# Optional Python ABI patch; only its shallow dataset metadata is searched.
WHEELS_PATCH_ROOT = None
patch_manifests = list(Path('/kaggle/input').glob('*/wheels_manifest.json'))
patch_manifests += list(Path('/kaggle/input').glob('datasets/*/*/wheels_manifest.json'))
patch_roots = [p.parent for p in patch_manifests
               if json.loads(p.read_text()).get('format') == 'rsna3-offline-wheels-patch-v1']
if len(patch_roots) > 1:
    raise ValueError(f'Attach a single wheels patch: {patch_roots}')
if patch_roots:
    WHEELS_PATCH_ROOT = patch_roots[0]
BOOTSTRAP_ROOT = WHEELS_PATCH_ROOT or BUNDLE_ROOT
module_spec = importlib.util.spec_from_file_location('kaggle_bootstrap', BOOTSTRAP_ROOT / 'kaggle_bootstrap.py')
bootstrap = importlib.util.module_from_spec(module_spec)
module_spec.loader.exec_module(bootstrap)
if WHEELS_PATCH_ROOT is None:
    bootstrap.install(BUNDLE_ROOT)
else:
    bootstrap.install(BUNDLE_ROOT, wheels_root=WHEELS_PATCH_ROOT)
WORK = Path('/kaggle/working')

from rsna3.common import read_json, require, atomic_json, sha256
from rsna3.inference import load_bundle
import shutil
SOURCE_BUNDLE = BUNDLE_ROOT
EXPORT = WORK / 'rsna3-inference'
require(not EXPORT.exists(), 'Use a fresh saved run: working inference bundle already exists.')
shutil.copytree(SOURCE_BUNDLE, EXPORT, ignore=shutil.ignore_patterns('__pycache__'))
manifest = load_bundle(EXPORT)
print('Fifteen exported model checkpoints and the training environment verified.')

def cli(*args):
    subprocess.run([sys.executable, '-m', 'rsna3', *map(str, args)], check=True)


In [ ]:
# Détection des données de compétition attachées, y compris les nouveaux chemins Kaggle.
COMPETITION_ROOT = None
if COMPETITION_ROOT is None:
    choices = [p.parent for p in Path('/kaggle/input').rglob('test_series.csv')
               if (p.parent / 'test.csv').exists()]
    if len(choices) != 1:
        raise ValueError(f'Choisir COMPETITION_ROOT parmi {choices}')
    COMPETITION_ROOT = choices[0]
COMPETITION_ROOT = Path(COMPETITION_ROOT)

def dicom_directory(split):
    choices = [COMPETITION_ROOT / name for name in (f'{split}_series', f'{split}_images', split)
               if (COMPETITION_ROOT / name).is_dir()]
    if len(choices) != 1:
        raise ValueError(f'Choisir le dossier DICOM {split} parmi {choices}')
    return choices[0]


In [ ]:
require(MEASURED_QUOTA_FACTOR > 0 and 0 < MAX_BENCHMARK_HOURS <= 2,
        'Use the measured positive quota factor and at most two hours for this benchmark.')
require(QUOTA_AVAILABLE - MAX_BENCHMARK_HOURS * MEASURED_QUOTA_FACTOR - .5 >= 8,
        'Keep eight quota hours protected, plus a thirty-minute setup allowance.')
import torch
require(torch.cuda.is_available(), 'Select GPU T4 x2 for the inference timing run.')
if 'admission_profile' in manifest:
    print('This exported bundle already contains an inference timing profile; reusing it.')
else:
    print('Timing DICOM decoding, rendering and all fifteen models on fifty studies.', flush=True)
    cli('benchmark-inference', '--bundle', EXPORT,
        '--studies-csv', COMPETITION_ROOT / 'train.csv',
        '--series-csv', COMPETITION_ROOT / 'train_series.csv',
        '--dicom-root', dicom_directory('train'),
        '--work', WORK / 'inference-benchmark', '--n', 50,
        '--quota-factor', MEASURED_QUOTA_FACTOR, '--max-hours', MAX_BENCHMARK_HOURS)


In [ ]:
# The benchmark updates manifest.json: refresh its offline checksum afterwards.
manifest = load_bundle(EXPORT)
profile = manifest.get('admission_profile')
require(profile is not None and profile.get('scope') == 'decode_render_all_15_models_global_ranks'
        and 50 <= profile.get('studies', 0) <= 100, 'A complete fifty-study timing profile is required.')
atomic_json(EXPORT / 'offline_manifest.json', {'format': 'rsna3-inference-offline-v1',
    'files': {str(p.relative_to(EXPORT)): sha256(p) for p in EXPORT.rglob('*')
              if p.is_file() and p.name != 'offline_manifest.json' and '__pycache__' not in p.parts}})
print(json.dumps({'stage': 'inference_timing_complete', 'exported_models': len(manifest['files']),
                  'admission_profile': profile, 'bundle_for_submission': str(EXPORT),
                  'next_step': 'Attach these saved outputs and the competition to 04_soumettre.ipynb.'}, indent=2))
